<a href="https://colab.research.google.com/github/ptsouth97/Training-Log-Dashboard/blob/main/Training_Log_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install Dependencies

In [ ]:
!pip install gspread google-auth plotly

In [ ]:
!pip install gspread google-auth

#Imports

In [12]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import gspread
from google.oauth2.service_account import Credentials

#Authenticate and Load Google Sheet

In [10]:
from google.colab import files

uploaded = files.upload()

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly"
]

creds = Credentials.from_service_account_file(
    list(uploaded.keys())[0],
    scopes=SCOPES
)

client = gspread.authorize(creds)

SPREADSHEET_ID = "1epQ4axsxFBJ4sTvhTiahAuDX1MqcWfj9iY7igJK5oHA"

sheet = client.open_by_key(SPREADSHEET_ID)
worksheet = sheet.worksheet("Log")

data = worksheet.get_all_values()

print("Loaded", len(data), "rows")

Saving training-log-dashboard-a3bb3322d539.json to training-log-dashboard-a3bb3322d539.json
Loaded 2532 rows


#Create DataFrame

In [13]:
headers = data[1]
rows = data[3:]

df = pd.DataFrame(rows, columns=headers)

df.head()

,DATE,DAYS,WEEK NUMBER,Weight (pounds),Mass (kg),Body Fat %,Muscle Mass %,Water Weight %,BMI (my calculation),Post run weight (pounds),...,Sodium (milligrams),Systolic,Diastolic,Heart rate,Temperature,Post workout carbs (grams - high),Post workout protein ideal (grams),Post workout protein - actual (grams),Runtypes,BMR calculation
0,7/15/2020,,,187.6,85.1,16.9,,,23.45,,...,157,,,,,141,39.3,24.2,Recovery,
1,7/16/2020,,,188.7,85.6,,,,23.59,,...,138,,,,,142,34.5,23.2,General aerobic,
2,7/17/2020,,,188.9,85.7,,,,23.61,,...,78,,,,,142,19.5,32.5,Threshold,
3,7/18/2020,,,189.4,85.9,,,,23.67,,...,164,,,,,142,40.9,33.8,Intervals,
4,7/19/2020,,,189.8,86.1,,,,23.72,,...,264,,,,,142,66.0,30.8,VO2 max,


#Data Cleaning

In [14]:
df["DATE"] = pd.to_datetime(
    df["DATE"],
    errors="coerce"
)

numeric_cols = [
    "Weight (pounds)",
    "HRV",
    "RHR",
    "Calories consumed",
    "Recovery score"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

df = df.sort_values("DATE")

#Basic Dashboard Metrics

In [15]:
metrics = {}

metrics["Weight"] = (
    df["Weight (pounds)"]
    .dropna()
    .iloc[-1]
)

metrics["HRV"] = (
    df["HRV"]
    .dropna()
    .iloc[-1]
)

metrics["RHR"] = (
    df["RHR"]
    .dropna()
    .iloc[-1]
)

metrics

{'Weight': np.float64(171.5), 'HRV': np.float64(48.0), 'RHR': np.float64(48.0)}

#Weight Trend

In [16]:
df["Weight_7D"] = (
    df["Weight (pounds)"]
    .rolling(7)
    .mean()
)

df["Weight_30D"] = (
    df["Weight (pounds)"]
    .rolling(30)
    .mean()
)

fig = px.line(
    df,
    x="DATE",
    y=[
        "Weight (pounds)",
        "Weight_7D",
        "Weight_30D"
    ],
    title="Weight Trend"
)

fig.show()

#HRV Trend

In [17]:
df["HRV_28D"] = (
    df["HRV"]
    .rolling(28)
    .mean()
)

fig = px.line(
    df,
    x="DATE",
    y=["HRV", "HRV_28D"],
    title="HRV vs Baseline"
)

fig.show()

#Recovery Dashboard

In [18]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["DATE"],
        y=df["HRV"],
        name="HRV"
    )
)

fig.add_trace(
    go.Scatter(
        x=df["DATE"],
        y=df["RHR"],
        name="RHR",
        yaxis="y2"
    )
)

fig.update_layout(
    title="HRV and Resting HR",
    yaxis2=dict(
        overlaying="y",
        side="right"
    )
)

fig.show()

#Correlation Heatmap

In [21]:
corr_cols = [
    "Weight (pounds)",
    "HRV",
    "RHR",
    "Calories consumed"
]

corr_df = df[corr_cols]

corr = corr_df.corr()

px.imshow(
    corr,
    text_auto=True,
    title="Correlation Matrix"
).show()

#Readiness Score

In [22]:
hrv_baseline = (
    df["HRV"]
    .rolling(28)
    .mean()
)

df["HRV_RATIO"] = (
    df["HRV"] /
    hrv_baseline
)

df["RHR_RATIO"] = (
    df["RHR"] /
    df["RHR"].rolling(28).mean()
)

df["READINESS"] = (
    60 * df["HRV_RATIO"] +
    40 * (2 - df["RHR_RATIO"])
)

df["READINESS"] = (
    df["READINESS"]
    .clip(0, 100)
)

df[[
    "DATE",
    "READINESS"
]].tail()

,DATE,READINESS
2524,2027-06-13,NaN
2525,2027-06-14,NaN
2526,2027-06-15,NaN
2527,2027-06-16,NaN
2528,2027-06-17,NaN


#Identify HRV Drivers

In [23]:
target = "HRV"

corrs = (
    df.corr(numeric_only=True)[target]
    .sort_values(ascending=False)
)

corrs.head(20)

,HRV
HRV,1.000000
HRV_28D,0.688162
HRV_RATIO,0.662662
READINESS,0.562176
Calories consumed,0.248544
Weight (pounds),-0.204635
Weight_30D,-0.212850
Weight_7D,-0.223929
RHR_RATIO,-0.528119
RHR,-0.858668


#Which Columns Have Data

In [25]:
summary = pd.DataFrame({
    "column": df.columns,
    "non_null": [df[c].replace('', np.nan).notna().sum() for c in df.columns]
})

# Explicitly convert the 'non_null' column to a numeric type to prevent ambiguity during sorting.
summary["non_null"] = pd.to_numeric(summary["non_null"], errors='coerce')

summary = summary.sort_values("non_null", ascending=False)

summary.head(100)

,column,non_null
0,DATE,2529.0
93,Carbs required (g),2410.0
96,Fat required (g),2410.0
98,Protein required (g),2410.0
92,Net calories,2364.0
...,...,...
73,Average GC Time Balance (right foot) %,512.0
16,VO2 max,482.0
100,Saturated Fat,402.0
67,Velo Score,382.0


#Find All Numeric Columns

In [27]:
numeric_cols = []

for i, col_name in enumerate(df.columns):
    # Select the column by integer position to ensure we get a Series,
    # even if column names are duplicated.
    series_to_process = df.iloc[:, i]

    s = pd.to_numeric(series_to_process, errors="coerce")

    if s.notna().sum() > 100:
        numeric_cols.append(col_name) # Append the original column name

print(f"{len(numeric_cols)} numeric columns found")

numeric_cols

87 numeric columns found


['DATE',
 'DAYS',
 'WEEK NUMBER',
 'Weight (pounds)',
 'Mass (kg)',
 'Body Fat %',
 'Muscle Mass %',
 'Water Weight %',
 'BMI (my calculation)',
 'Recovery',
 'Day Strain',
 'Calories',
 'HRV',
 'RHR',
 'Respiratory Rate',
 'VO2 max',
 'Sleep Performance',
 'WHOOP Avg HR (bpm)',
 'Max HR',
 'WHOOP Activity Calories Burned',
 'WHOOP Activity Strain',
 'Elevation (ft)',
 'Temperature (F)',
 'Humidity (%)',
 'Feels like',
 'Distance (miles)',
 'GARMIN Average HR (bpm)',
 'Average Cadence (spm)',
 'Average Power (W)',
 'Functional Threshold Power (rFTPw)',
 'Average Vertical Oscillation (cm)',
 'Average Ground Contact Time (ms)',
 'RSS',
 'Cycling Distance (miles)',
 'Cycling Elevation (ft)',
 'Calories',
 'Average ZWIFT Power',
 'Average Cycling HR',
 'Average Cycling Cadence',
 'Normalized Power',
 'Trainer Road TSS',
 'Trainer Road FTP (W)',
 'Last 6 weeks Strava estimated FTP (W)',
 'zFTP (W)',
 'Normalized zFTP (W/kg)',
 'zMAP (W)',
 'zVO2 max',
 'Zwift Racing Score',
 'Velo2 Race Sco

#Create A Full Correlation Matrix

In [30]:
all_numeric_series = []
numeric_col_name_counts = {}

# Iterate through each column in the original DataFrame `df` by index.
# This ensures we get individual Series, even if column names are duplicated.
for i, col_name in enumerate(df.columns):
    # Check if this column's name was identified as numeric.
    if col_name in numeric_cols:
        series_to_convert = df.iloc[:, i]

        # Generate a unique name for this column in the new DataFrame `numeric_df`.
        # If a name is duplicated, append a counter (e.g., 'Calories', 'Calories.1').
        if col_name in numeric_col_name_counts:
            numeric_col_name_counts[col_name] += 1
            final_col_name = f"{col_name}.{numeric_col_name_counts[col_name]}"
        else:
            numeric_col_name_counts[col_name] = 0
            final_col_name = col_name

        # Convert the Series to numeric and add it to our list, renaming it for uniqueness.
        all_numeric_series.append(pd.to_numeric(series_to_convert, errors="coerce").rename(final_col_name))

# Concatenate all processed Series into the final `numeric_df`.
numeric_df = pd.concat(all_numeric_series, axis=1)

corr = numeric_df.corr()

In [31]:
px.imshow(
    corr,
    title="Full Correlation Matrix",
    aspect="auto"
).show()

#Top HRV Predictors

In [32]:
corr["HRV"].sort_values(
    ascending=False
).head(20)

,HRV
HRV,1.000000
Garmin HRV Overnight Avg,0.886945
HRV_28D,0.688162
Body Battery Maximum,0.680926
HRV_RATIO,0.662662
Recovery,0.611526
READINESS,0.562176
Running VO2 max,0.448481
Protein (g) / Pound of weight,0.337638
LOSE IT budget,0.337367


In [33]:
corr["HRV"].sort_values().head(20)

,HRV
RHR,-0.858668
Garmin RHR,-0.749349
Carbs per hour,-0.698600
RHR_RATIO,-0.528119
Respiratory Rate,-0.460612
Average Vertical Oscillation (cm),-0.370234
zVO2 max,-0.350618
zMAP (W),-0.348430
Trainer Road FTP (W),-0.315567
Last 6 weeks Strava estimated FTP (W),-0.292143


#Top Weight Predictors

In [34]:
corr["Weight (pounds)"]\
    .sort_values()\
    .head(20)

,Weight (pounds)
Muscle Mass %,-0.954789
LOSE IT budget,-0.710781
Functional Threshold Power (rFTPw),-0.658173
VO2 max,-0.654746
Running VO2 max,-0.599744
Velo2 Race Score,-0.506120
Protein actual (g),-0.495326
Calories,-0.438219
Day Strain,-0.432878
Protein (g) / Pound of weight,-0.401585


#Look At Recent Trends

In [35]:
def rolling_chart(column):

    temp = df.copy()

    temp[column] = pd.to_numeric(
        temp[column],
        errors="coerce"
    )

    temp["7D"] = temp[column].rolling(7).mean()
    temp["28D"] = temp[column].rolling(28).mean()

    fig = px.line(
        temp,
        x="DATE",
        y=[column, "7D", "28D"],
        title=column
    )

    fig.show()

In [36]:
rolling_chart("HRV")
rolling_chart("RHR")
rolling_chart("Weight (pounds)")
rolling_chart("Recovery")
rolling_chart("Running VO2 max")